In [1]:
import originpro as op
import os

# ===== 1) 打开工程并确认是否成功 =====
src_opju = r"C:\usrspace\mywork\edges.opju"   # <— 用绝对路径；注意路径和文件名是否正确
print("File exists? ", os.path.exists(src_opju))
ok = op.open(file=src_opju, readonly=True)
print("Project opened? ", ok)

# ===== 2) 列出所有页面类型，看看有没有工作簿 =====
wbooks = list(op.pages('w'))   # 所有工作簿（Worksheet Book）
mbks   = list(op.pages('m'))   # 所有矩阵簿（Matrix Book）
graphs = list(op.pages('g'))   # 所有图页（Graph），仅排查
notes  = list(op.pages('n'))   # 所有Notes，仅排查

print("#Workbooks:", len(wbooks), "| #MatrixBooks:", len(mbks), "| #Graphs:", len(graphs), "| #Notes:", len(notes))

# ===== 3) 如果有工作簿：打印每个工作簿下所有工作表名 =====
for wb in wbooks:
    sheet_names = [wks.name for wks in wb]
    print(f"[{wb.name}] -> {sheet_names}")

# ===== 4) 如果是矩阵簿：打印矩阵表名（有些项目只有矩阵簿，没有工作簿）=====
for mb in mbks:
    ms_names = [ms.name for ms in mb]
    print(f"[{mb.name}] (Matrix) -> {ms_names}")

# ===== 5) 若你想强行拿到第一个工作簿并遍历 =====
if wbooks:
    wb = wbooks[0]
    print("Active to:", wb.name)
    wb.activate()  # 让它成为激活窗口
    for wks in wb:
        print("Sheet:", wks.name)



File exists?  True
Project opened?  True
#Workbooks: 1 | #MatrixBooks: 0 | #Graphs: 37 | #Notes: 0
[Book1] -> ['pending_edges_ttb30', 'pending_edges_ttb40', 'pending_edges_ttb50', 'pending_edges_ttb60', 'pending_edges_ttb70', 'pending_edges_ttb80', 'pending_edges_ttb90', 'pending_edges_ttb100', 'pending_edges_ttb110', 'pending_edges_ttb120', 'pending_edges_ttb130', 'pending_edges_ttb140']
Active to: Book1
Sheet: pending_edges_ttb30
Sheet: pending_edges_ttb40
Sheet: pending_edges_ttb50
Sheet: pending_edges_ttb60
Sheet: pending_edges_ttb70
Sheet: pending_edges_ttb80
Sheet: pending_edges_ttb90
Sheet: pending_edges_ttb100
Sheet: pending_edges_ttb110
Sheet: pending_edges_ttb120
Sheet: pending_edges_ttb130
Sheet: pending_edges_ttb140


In [8]:
wks = op.find_sheet('w', '[Book1]pending_edges_ttb100')

In [9]:
wks.activate()  # 可选：激活这张表

1

In [12]:
print("形状 (rows, cols):", wks.shape)               # 来自 DSheet 的 shape 属性


形状 (rows, cols): (21894, 2)


In [13]:
df = wks.to_df(head='L')     # 也可以 head='C' 用注释行命名，或 head='' 用短名 A,B,C...
print(df.head())
print(df.dtypes)

   time  pending_edges
0   0.0           47.0
1   1.0           47.0
2   2.0           47.0
3   3.0           47.0
4   4.0           47.0
time             float64
pending_edges    float64
dtype: object


In [18]:
mapping = df.set_index('time')['pending_edges'].to_dict()

In [29]:
# 假设 mapping: {time -> edge}
t_min = int(min(mapping))
t_max = int(max(mapping))
size = t_max - t_min + 1
edges = [None] * size                       # 预分配
for t, e in mapping.items():
    ti = int(t) - t_min                     # 用“偏移”作为索引
    edges[ti] = e
# edges[k] 就对应真实时间 t = t_min + k


In [31]:
best_len = 0
best_start_idx = None

cur_len = 0
cur_start_idx = None

for i, val in enumerate(edges):
    # 把 None 当成“中断”；只把严格等于 0 计入连续段
    if val == 0:
        if cur_len == 0:
            cur_start_idx = i
        cur_len += 1
        if cur_len > best_len:
            best_len = cur_len
            best_start_idx = cur_start_idx
    else:
        cur_len = 0
        cur_start_idx = None

if best_len > 0:
    start_t = t_min + best_start_idx
    end_t   = start_t + best_len - 1
    dur_s   = best_len
    dur_min = dur_s / 60.0
    print(f"Longest zero-edge window: [{start_t}, {end_t}]  "
          f"len={dur_s}s (~{dur_min:.2f} min)")
else:
    print("No zero-edge window found.")


Longest zero-edge window: [16411, 16498]  len=88s (~1.47 min)
